In [0]:
# if geopands is not installed run : !pip install geopandas

**Neighborhood Data**

In [0]:
from pathlib import Path
import geopandas as gpd

BASE_DIR = Path.cwd().parent
GEO_DIR = BASE_DIR / "data" / "raw" / "geo"

gdf = gpd.read_file(GEO_DIR / "Neighborhoods.shp")

print(gdf.shape)
gdf.plot()

In [0]:
print(len(gdf))  # should be 79

In [0]:
extras = [
    "Bellefontaine/Calvary Cemetery",
    "Carondelet Park",
    "Fairground Park",
    "Forest Park",
    "Missouri Botanical Garden",
    "O'Fallon Park",
    "Penrose Park",
    "Tower Grove Park",
    "Willmore Park"
]

In [0]:
gdf_clean = gdf[~gdf["NHD_NAME"].isin(extras)].copy()

print("rows after filter:", len(gdf_clean))
print("unique names after filter:", gdf_clean["NHD_NAME"].nunique())

In [0]:
gdf_clean = gdf_clean.to_crs("EPSG:4326")

In [0]:
gdf_clean.to_file(GEO_DIR / "stl_neighborhoods.geojson", driver="GeoJSON")

In [0]:
if len(gdf_clean) != 79:
    raise ValueError(f"Expected 79 neighborhoods, found {len(gdf_clean)}")

**FEMA Data**

In [0]:
fema_gdf = gpd.read_file(GEO_DIR / "S_FLD_HAZ_AR.shp")

In [0]:
print("rows:", len(fema_gdf))
print("columns:", fema_gdf.columns)
print("crs:", fema_gdf.crs)

In [0]:
fema_gdf = fema_gdf.to_crs("EPSG:4326")

In [0]:
gdf_clean.to_file(GEO_DIR /"fema_flood_zones.geojson",
    driver="GeoJSON"
)

In [0]:
print(fema_gdf.columns)

In [0]:
print("Neighborhood CRS:", gdf_clean.crs)
print("FEMA CRS:", fema_gdf.crs)

In [0]:
tracts_gdf = gpd.read_file(GEO_DIR/"tl_2025_29_tract.shp")

In [0]:
print(tracts_gdf.columns)
print(len(tracts_gdf))
print(tracts_gdf.crs)

In [0]:
tracts_gdf["COUNTYFP"].unique()

In [0]:
tracts_gdf = tracts_gdf[
    tracts_gdf["COUNTYFP"].isin(["510", "189"])
]

In [0]:
tracts_gdf = tracts_gdf.to_crs("EPSG:4326")

In [0]:
print("tract CRS:", tracts_gdf.crs)

In [0]:
projected_crs = "EPSG:26915"

tracts_proj = tracts_gdf.to_crs(projected_crs).copy()
neighborhoods_proj = gdf_clean.to_crs(projected_crs).copy()

In [0]:
tracts_proj["tract_area"] = tracts_proj.geometry.area

In [0]:
overlay_gdf = gpd.overlay(tracts_proj, neighborhoods_proj, how="intersection")

In [0]:
overlay_gdf["intersect_area"] = overlay_gdf.geometry.area
overlay_gdf["area_weight"] = overlay_gdf["intersect_area"] / overlay_gdf["tract_area"]

In [0]:
overlay_gdf = overlay_gdf.to_crs("EPSG:4326")

In [0]:
print("overlay rows:", len(overlay_gdf))
print(overlay_gdf.columns)

overlay_gdf[["GEOID", "NHD_NAME", "tract_area", "intersect_area", "area_weight"]].head()

In [0]:
overlay_gdf.to_file(GEO_DIR / "tract_neighborhood_overlay.geojson", driver="GeoJSON")

In [0]:
overlay.head()
overlay.plot(
    column="area_weight",
    cmap="viridis",
    legend=True,
    figsize=(10,10)
)

This map shows how much of each St. Louis neighborhood’s area falls within FEMA-designated flood zones. It was created using an area-weighted spatial overlay between FEMA flood hazard polygons and neighborhood boundaries, where we calculated the percentage of each neighborhood’s land that intersects flood zones. Lighter colors (yellow/green) represent neighborhoods with higher flood exposure, while darker colors (blue/purple) indicate lower exposure.

In [0]:
# Saved in Google Colab but they already exist in the repo
# from google.colab import files

#files.download("data/raw/geo/stl_neighborhoods.geojson")
#files.download("data/raw/geo/fema_flood_zones.geojson")
#files.download("data/raw/geo/tract_neighborhood_overlay.geojson")